# M01D · Practice Problems: Applied Computational Biology Problem Set (Solutions)
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M01_Python_Basics/solutions/M01D_Practice_Problems_Solutions.ipynb)

⏱ ~150 min · Difficulty: ★★☆ to ★★★ · Prerequisites: M01A, M01B, M01C


## 1. Learning Outcomes

By the end of this lab session problem set, you will be able to:
1. **Apply** core Python data structures (`list`, `dict`, `set`, `tuple`) and control flow to realistic computational biology challenges without starter scaffolding.
2. **Implement** signature motif classification algorithms using regular expressions and string matching.
3. **Compute** evolutionary substitution rates, transition/transversion ($T_s / T_v$) ratios, and frameshift mutation consequences.
4. **Construct** position frequency matrices (PFMs) and derive majority-rule consensus sequences from aligned motifs.
5. **Scan** genomic sequences for palindromic restriction endonuclease recognition sites and trinucleotide repeat expansion disorders.
6. **Calculate** codon usage bias metrics (RSCU) and protein hydropathy profiles using established biochemical scales.


## 2. Prerequisites

- Thorough mastery of **M01A (Core Python)**, **M01B (Sequence Manipulation)**, and **M01C (ORFs, Exons & Introns)**.
- Comfort with writing multi-step functions, string slicing, dictionary accumulation, and nested loops.


## 3. Setup

Run this cell to detect the runtime environment, configure paths to `data/` and `outputs/`, and initialize directories.


In [ ]:
# --- Setup -------------------------------------------------------
import sys
import os
import platform
import re
from collections import Counter
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Cloning repository...")
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    p = Path.cwd()
    found_root = None
    for candidate in [p] + list(p.parents):
        if (candidate / "data" / "sequences").exists() or (candidate / "COURSE_PLAN.md").exists():
            found_root = candidate
            break
    if found_root:
        ROOT = found_root
    else:
        repo_candidate = Path("/Users/prateekpaul/Library/CloudStorage/GoogleDrive-prateekp@iiitd.ac.in/My Drive/IIITD/mygithub/ProgrammingCB")
        ROOT = repo_candidate if repo_candidate.exists() else p

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "M01D"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Python version     : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"✓ Repository Root    : {ROOT.resolve()}")
print(f"✓ Data Directory     : {DATA_DIR.resolve()}")
print(f"✓ Output Directory   : {OUT_DIR.resolve()}")


## 4. Problem Set Instructions

This notebook is an intensive, self-contained computational laboratory with 15 problems spanning protein biochemistry, genome annotation, and evolutionary genetics.


In [ ]:
CODON_TABLE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
    'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
}

_REV_TRANS = str.maketrans("ACGTUacgtuNn", "TGCAAtgcaaNn")
def reverse_complement(seq: str) -> str:
    return seq.translate(_REV_TRANS)[::-1]


### Problem 1 (Level 1 ★): Protein Sequence Validation and Net Charge at Physiological pH 7.4

Validates 20 standard amino acids and calculates net charge: basic (K, R, H = +1) vs acidic (D, E = -1).

- **Input**: Protein string `seq`.
- **Expected Output**: Dictionary with `'is_valid'`, `'net_charge'`, `'pos_count'`, `'neg_count'`.


In [ ]:
def analyze_protein_charge(seq: str) -> dict:
    clean = seq.upper().strip()
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if not clean or any(aa not in valid_aa for aa in clean):
        return {"is_valid": False, "net_charge": 0, "pos_count": 0, "neg_count": 0}
    pos = sum(1 for aa in clean if aa in {"K", "R", "H"})
    neg = sum(1 for aa in clean if aa in {"D", "E"})
    return {"is_valid": True, "net_charge": pos - neg, "pos_count": pos, "neg_count": neg}


#### 💡 Solution Explanation
Tallies basic and acidic residues at pH 7.4.

In [ ]:
res = analyze_protein_charge('MKHDEAD')
assert res['net_charge'] == -1
assert res['is_valid'] is True
print('✓ Problem 1 passed!')


### Problem 2 (Level 1 ★): Enzyme Class Motif Classifier

Classifies protein into Serine Protease (`G.S.G`), Zinc Finger (`C.{2,4}C.{12}H.{3,5}H`), Kinase (`[LIVMFY]...[RK]`), Aspartyl Protease (`[DS]TG[ST]`), or Unclassified.

- **Input**: Protein string `seq`.
- **Expected Output**: String classification.


In [ ]:
def classify_enzyme(seq: str) -> str:
    clean = seq.upper()
    if re.search(r"G.S.G", clean): return "Serine Protease"
    if re.search(r"C.{2,4}C.{12}H.{3,5}H", clean): return "Zinc Finger"
    if re.search(r"[LIVMFY]...[RK]", clean): return "Protein Kinase"
    if re.search(r"[DS]TG[ST]", clean): return "Aspartyl Protease"
    return "Unclassified"


#### 💡 Solution Explanation
Regular expression search for PROSITE catalytic signatures.

In [ ]:
assert classify_enzyme('AAAGASAGAAA') == 'Serine Protease'
assert classify_enzyme('LAAARK') == 'Protein Kinase'
print('✓ Problem 2 passed!')


### Problem 3 (Level 2 ★★): Sliding Window GC Content Profiler

Calculates sliding window GC% returning list of `(midpoint_coord, gc_pct)` tuples.

- **Input**: String `seq`, integers `window_size`, `step`.
- **Expected Output**: List of `(mid, gc)` tuples.


In [ ]:
def sliding_window_gc_profile(seq: str, window_size: int, step: int) -> list[tuple[int, float]]:
    clean = seq.upper()
    profile = []
    for i in range(0, len(clean) - window_size + 1, step):
        win = clean[i:i+window_size]
        gc = win.count("G") + win.count("C")
        profile.append((i + (window_size // 2), round((gc / window_size) * 100.0, 2)))
    return profile


#### 💡 Solution Explanation
Strided iteration computing midpoint and GC percentage.

In [ ]:
res = sliding_window_gc_profile('GCGCATATGCGC', 4, 4)
assert res == [(2, 100.0), (6, 0.0), (10, 100.0)]
print('✓ Problem 3 passed!')


### Problem 4 (Level 2 ★★): Most Frequent K-mers across Length Range

For each $k \in [k_{\min}, k_{\max}]$, finds all k-mers achieving maximum count, sorted alphabetically.

- **Input**: String `seq`, integers `k_min`, `k_max`.
- **Expected Output**: Dictionary mapping `k: [(kmer, max_count), ...]`.


In [ ]:
def most_frequent_kmers_range(seq: str, k_min: int, k_max: int) -> dict[int, list[tuple[str, int]]]:
    clean = seq.upper()
    results = {}
    for k in range(k_min, k_max + 1):
        if len(clean) < k:
            results[k] = []
            continue
        counts = {}
        for i in range(len(clean) - k + 1):
            km = clean[i:i+k]
            counts[km] = counts.get(km, 0) + 1
        if not counts:
            results[k] = []
            continue
        max_c = max(counts.values())
        top_kmers = sorted([km for km, c in counts.items() if c == max_c])
        results[k] = [(km, max_c) for km in top_kmers]
    return results


#### 💡 Solution Explanation
Finds maximal count for each length $k$ and extracts tied motifs.

In [ ]:
res = most_frequent_kmers_range('ACACACGT', 2, 3)
assert res[2] == [('AC', 3)]
print('✓ Problem 4 passed!')


### Problem 5 (Level 2 ★★): Non-Overlapping DNA Motif Scanner

Finds all 0-based start positions of non-overlapping occurrences of `motif` in `seq`.

- **Input**: Strings `seq` and `motif`.
- **Expected Output**: List of start indices.


In [ ]:
def find_non_overlapping_motifs(seq: str, motif: str) -> list[int]:
    s, m = seq.upper(), motif.upper()
    if not m or len(s) < len(m): return []
    pos, start, m_len = [], 0, len(m)
    while True:
        idx = s.find(m, start)
        if idx == -1: break
        pos.append(idx)
        start = idx + m_len
    return pos


#### 💡 Solution Explanation
Jumps past each matched motif (`start = idx + len(m)`).

In [ ]:
assert find_non_overlapping_motifs('AAAAAA', 'AA') == [0, 2, 4]
print('✓ Problem 5 passed!')


### Problem 6 (Level 2 ★★): Transition vs Transversion Ratio (Ts/Tv) Calculator

Calculates transitions ($A \leftrightarrow G, C \leftrightarrow T$), transversions, and $T_s / T_v$ ratio.

- **Input**: Two equal-length DNA strings.
- **Expected Output**: Dictionary with counts and ratio.


In [ ]:
def calculate_ts_tv_ratio(s1: str, s2: str) -> dict:
    if len(s1) != len(s2): raise ValueError("Unequal lengths!")
    pur, pyr = {'A', 'G'}, {'C', 'T'}
    ts, tv = 0, 0
    for b1, b2 in zip(s1.upper(), s2.upper()):
        if b1 == b2: continue
        if (b1 in pur and b2 in pur) or (b1 in pyr and b2 in pyr):
            ts += 1
        else:
            tv += 1
    r = round(ts / tv, 2) if tv > 0 else float('inf')
    return {"transitions": ts, "transversions": tv, "ratio": r}


#### 💡 Solution Explanation
Pairwise substitution classification comparing purine and pyrimidine rings.

In [ ]:
res = calculate_ts_tv_ratio('ACA', 'GTC')
assert res['transitions'] == 2 and res['transversions'] == 1 and res['ratio'] == 2.0
print('✓ Problem 6 passed!')


### Problem 7 (Level 2 ★★): Palindromic Restriction Site Scanner

Finds all reverse-complement palindromes of specified lengths in a DNA string.

- **Input**: DNA string and tuple of lengths.
- **Expected Output**: List of `{'position', 'length', 'site'}` dictionaries.


In [ ]:
def find_restriction_palindromes(seq: str, lengths: tuple[int, ...] = (4, 6)) -> list[dict]:
    clean = seq.upper()
    pals = []
    for L in lengths:
        for i in range(len(clean) - L + 1):
            sub = clean[i:i+L]
            if sub == reverse_complement(sub):
                pals.append({"position": i, "length": L, "site": sub})
    return sorted(pals, key=lambda p: (p["position"], p["length"]))


#### 💡 Solution Explanation
Identifies restriction sites where `sub == reverse_complement(sub)`.

In [ ]:
pals = find_restriction_palindromes('GAATTC', (4, 6))
sites = [(p['position'], p['length'], p['site']) for p in pals]
assert (0, 6, 'GAATTC') in sites
print('✓ Problem 7 passed!')


### Problem 8 (Level 3 ★★★): Frameshift Mutation Detector

Detects indel mutation divergence, classifies frameshift vs in-frame, and translates both sequences.

- **Input**: Wildtype and mutant DNA strings.
- **Expected Output**: Diagnostic dictionary.


In [ ]:
def detect_frameshift_mutation(wildtype_dna: str, mutant_dna: str) -> dict:
    wt, mut = wildtype_dna.upper(), mutant_dna.upper()
    min_len = min(len(wt), len(mut))
    pos = min_len
    for i in range(min_len):
        if wt[i] != mut[i]:
            pos = i
            break
    diff = abs(len(wt) - len(mut))
    is_fs = (diff % 3 != 0)
    def trans(s):
        p = []
        for k in range(0, len(s) - 2, 3):
            aa = CODON_TABLE.get(s[k:k+3], "X")
            if aa == "*": break
            p.append(aa)
        return "".join(p)
    return {"mutation_pos": pos, "length_diff": diff, "is_frameshift": is_fs, "wt_protein": trans(wt), "mut_protein": trans(mut)}


#### 💡 Solution Explanation
Pinpoints divergence position and evaluates triplet frame disruption.

In [ ]:
res = detect_frameshift_mutation('ATGGCCATTTGA', 'ATGCCATTTGA')
assert res['is_frameshift'] is True
assert res['wt_protein'] == 'MAI'
assert res['mut_protein'] == 'MPF'
print('✓ Problem 8 passed!')


### Problem 9 (Level 3 ★★★): Position Frequency Matrix (PFM) and Consensus Sequence Generator

Constructs 4xL PFM dictionary and derives majority-rule consensus sequence with alphabetical tiebreak.

- **Input**: List of equal-length DNA strings.
- **Expected Output**: Dictionary with `'pfm'` and `'consensus'`.


In [ ]:
def generate_pfm_and_consensus(aligned_seqs: list[str]) -> dict:
    if not aligned_seqs: return {"pfm": {}, "consensus": ""}
    L = len(aligned_seqs[0])
    bases = ["A", "C", "G", "T"]
    pfm = {b: [0] * L for b in bases}
    for s in aligned_seqs:
        for idx, char in enumerate(s.upper()):
            if char in pfm: pfm[char][idx] += 1
    cons = []
    for col in range(L):
        best = max(bases, key=lambda b: (pfm[b][col], -ord(b)))
        cons.append(best)
    return {"pfm": pfm, "consensus": "".join(cons)}


#### 💡 Solution Explanation
Tallies base counts per column and selects majority consensus.

In [ ]:
res = generate_pfm_and_consensus(['ATGC', 'ATCC', 'AGCC', 'ATGC'])
assert res['consensus'][:2] == 'AT'
print('✓ Problem 9 passed!')


### Problem 10 (Level 3 ★★★): Comprehensive 6-Frame ORF Finder

Scans all 6 frames for ORFs starting with ATG and ending with in-frame stop with length >= min_aa.

- **Input**: DNA string and integer `min_aa_length`.
- **Expected Output**: List of ORF dictionaries sorted by length desc.


In [ ]:
def find_all_6frame_orfs(dna_seq: str, min_aa_length: int = 20) -> list[dict]:
    clean_fwd = dna_seq.upper()
    clean_rev = reverse_complement(clean_fwd)
    stops = {"TAA", "TAG", "TGA"}
    all_orfs = []
    def scan_strand(seq_s, sign):
        for off in (0, 1, 2):
            i = off
            while i < len(seq_s) - 2:
                if seq_s[i:i+3] == "ATG":
                    st = i
                    for j in range(st + 3, len(seq_s) - 2, 3):
                        if seq_s[j:j+3] in stops:
                            orf_nt = seq_s[st:j]
                            prot = "".join(CODON_TABLE.get(orf_nt[k:k+3], 'X') for k in range(0, len(orf_nt), 3))
                            if len(prot) >= min_aa_length:
                                all_orfs.append({"strand": sign, "frame": f"{sign}{off+1}", "length_aa": len(prot), "protein": prot})
                            break
                i += 3
    scan_strand(clean_fwd, "+")
    scan_strand(clean_rev, "-")
    return sorted(all_orfs, key=lambda o: -o["length_aa"])


#### 💡 Solution Explanation
6-frame genome annotation scan.

In [ ]:
test_dna = 'NNNN' + 'ATG' + ('GCC'*20) + 'TAA' + 'NNNN'
orfs = find_all_6frame_orfs(test_dna, min_aa_length=20)
assert len(orfs) >= 1 and orfs[0]['length_aa'] == 21
print('✓ Problem 10 passed!')


### Problem 11 (Level 2 ★★): Kyte-Doolittle Hydrophobicity Profile

Calculates protein hydropathy profile using Kyte-Doolittle scale in sliding windows of size $W$. Return list of `(center_pos, mean_hydropathy)` rounded to 2 decimals. Scale: I:4.5, V:4.2, L:3.8, F:2.8, C:2.5, M:1.9, A:1.8, G:-0.4, T:-0.7, S:-0.8, W:-0.9, Y:-1.3, P:-1.6, H:-3.2, E:-3.5, Q:-3.5, D:-3.5, N:-3.5, K:-3.9, R:-4.5.

- **Input**: Protein sequence string and integer `window_size` (odd).
- **Expected Output**: List of `(midpoint, mean_score)` tuples.


In [ ]:
def compute_hydropathy_profile(protein_seq: str, window_size: int = 9) -> list[tuple[int, float]]:
    scale = {
        'I': 4.5, 'V': 4.2, 'L': 3.8, 'F': 2.8, 'C': 2.5, 'M': 1.9, 'A': 1.8,
        'G': -0.4, 'T': -0.7, 'S': -0.8, 'W': -0.9, 'Y': -1.3, 'P': -1.6,
        'H': -3.2, 'E': -3.5, 'Q': -3.5, 'D': -3.5, 'N': -3.5, 'K': -3.9, 'R': -4.5
    }
    clean = protein_seq.upper()
    if len(clean) < window_size: return []
    profile = []
    half = window_size // 2
    for i in range(len(clean) - window_size + 1):
        win = clean[i:i+window_size]
        avg = sum(scale.get(aa, 0.0) for aa in win) / window_size
        profile.append((i + half, round(avg, 2)))
    return profile


#### 💡 Solution Explanation
Identifies hydrophobic transmembrane helices (scores > +1.6 indicate potential membrane spans).

In [ ]:
pep = 'IIIIIKKKKK'
prof = compute_hydropathy_profile(pep, window_size=5)
assert len(prof) == 6
assert prof[0][1] == 4.5 # window of all I
print('✓ Problem 11 passed!')


### Problem 12 (Level 2 ★★): Trinucleotide Repeat Expansion Detector (Huntington's Disease)

Finds the maximum length run of tandem repeats of a trinucleotide (e.g. `'CAG'`) and classifies pathogenicity (Huntington's: normal <= 35, pathogenic >= 36). Return `{'repeat_unit': str, 'max_repeats': int, 'is_pathogenic': bool}`.

- **Input**: DNA string `seq`, repeat string `unit`, integer `threshold`.
- **Expected Output**: Diagnostic dictionary.


In [ ]:
def detect_triplet_repeats(seq: str, unit: str = 'CAG', pathogenic_threshold: int = 36) -> dict:
    s = seq.upper()
    u = unit.upper()
    k = len(u)
    max_reps = 0
    current_reps = 0
    i = 0
    while i <= len(s) - k:
        if s[i:i+k] == u:
            current_reps += 1
            if current_reps > max_reps:
                max_reps = current_reps
            i += k
        else:
            current_reps = 0
            i += 1
    return {'repeat_unit': u, 'max_repeats': max_reps, 'is_pathogenic': max_reps >= pathogenic_threshold}


#### 💡 Solution Explanation
Scans for tandem microsatellite repeat expansions diagnostic of neurodegenerative trinucleotide disorders.

In [ ]:
dna = 'NNN' + ('CAG' * 40) + 'NNN'
res = detect_triplet_repeats(dna, 'CAG', 36)
assert res['max_repeats'] == 40
assert res['is_pathogenic'] is True
print('✓ Problem 12 passed!')


### Problem 13 (Level 3 ★★★): Relative Synonymous Codon Usage (RSCU) Calculator

Computes RSCU for each codon in a CDS: $RSCU = \frac{X_{ij}}{\frac{1}{n_i} \sum_{j=1}^{n_i} X_{ij}}$ where $X_{ij}$ is observed count of codon $j$ for amino acid $i$, and $n_i$ is number of synonymous codons for that amino acid. Return dictionary `{codon: rscu}` rounded to 2 decimals.

- **Input**: Coding sequence string.
- **Expected Output**: Dictionary of float RSCU values.


In [ ]:
def calculate_rscu(cds: str) -> dict[str, float]:
    clean = cds.upper()
    codons = [clean[i:i+3] for i in range(0, len(clean) - 2, 3) if len(clean[i:i+3]) == 3]
    counts = Counter(codons)
    # Group codons by amino acid
    aa_to_codons = {}
    for cod, aa in CODON_TABLE.items():
        if aa != "*":
            aa_to_codons.setdefault(aa, []).append(cod)
    rscu = {}
    for aa, syn_codons in aa_to_codons.items():
        tot_syn = sum(counts.get(c, 0) for c in syn_codons)
        n = len(syn_codons)
        for c in syn_codons:
            if tot_syn == 0:
                rscu[c] = 1.0 # Equal usage by default
            else:
                expected = tot_syn / n
                rscu[c] = round(counts.get(c, 0) / expected, 2)
    return rscu


#### 💡 Solution Explanation
Calculates codon usage bias: RSCU > 1.0 indicates over-represented preferred codons; RSCU < 1.0 indicates rare codons.

In [ ]:
# If all Glutamate are 'GAA' and none are 'GAG', GAA has RSCU=2.0 and GAG has RSCU=0.0
cds = 'ATG' + ('GAA' * 10) + 'TAA'
rscu = calculate_rscu(cds)
assert rscu['GAA'] == 2.0
assert rscu['GAG'] == 0.0
print('✓ Problem 13 passed!')


### Problem 14 (Level 3 ★★★): Multi-Sample Pairwise Distance Matrix

Given a dictionary `{sample_id: sequence}`, computes pairwise Hamming distance matrix and returns `{'matrix': dict[str, dict[str, int]], 'closest_pair': tuple[str, str, int]}`.

- **Input**: Dictionary mapping sample IDs to equal-length strings.
- **Expected Output**: Matrix dictionary and closest pair.


In [ ]:
def pairwise_distance_matrix(sample_dict: dict[str, str]) -> dict:
    ids = list(sample_dict.keys())
    matrix = {s1: {s2: 0 for s2 in ids} for s1 in ids}
    closest_pair = None
    min_dist = float('inf')
    
    for i in range(len(ids)):
        for j in range(len(ids)):
            id1, id2 = ids[i], ids[j]
            seq1, seq2 = sample_dict[id1].upper(), sample_dict[id2].upper()
            dist = sum(1 for a, b in zip(seq1, seq2) if a != b)
            matrix[id1][id2] = dist
            if i < j and dist < min_dist:
                min_dist = dist
                closest_pair = (id1, id2, dist)
                
    return {'matrix': matrix, 'closest_pair': closest_pair}


#### 💡 Solution Explanation
Builds an $N \times N$ distance matrix for phylogenetics and clusters closest viral/bacterial isolates.

In [ ]:
samples = {'A': 'ATGC', 'B': 'ATGT', 'C': 'CCCC'}
res = pairwise_distance_matrix(samples)
assert res['matrix']['A']['B'] == 1
assert res['closest_pair'] == ('A', 'B', 1)
print('✓ Problem 14 passed!')


### Problem 15 (Level 3 ★★★): Regulatory Motif Search Allowing Mismatches

Finds all 0-based start positions in `seq` matching `motif` with at most `max_mismatches` differences. Return list of tuples `[(start_pos, mismatch_count, matched_seq), ...]`.

- **Input**: String `seq`, target `motif`, integer `max_mismatches`.
- **Expected Output**: List of matching tuples.


In [ ]:
def find_motifs_with_mismatches(seq: str, motif: str, max_mismatches: int = 1) -> list[tuple[int, int, str]]:
    s = seq.upper()
    m = motif.upper()
    k = len(m)
    matches = []
    for i in range(len(s) - k + 1):
        window = s[i:i+k]
        mismatches = sum(1 for a, b in zip(window, m) if a != b)
        if mismatches <= max_mismatches:
            matches.append((i, mismatches, window))
    return matches


#### 💡 Solution Explanation
Fuzzy motif search accommodating natural evolutionary divergence and transcription factor degeneracy.

In [ ]:
# Motif 'TATA' allowing 1 mismatch:
# 'TATA' (0 mismatches), 'TATG' (1 mismatch)
seq = 'TATAGCTATG'
res = find_motifs_with_mismatches(seq, 'TATA', max_mismatches=1)
assert (0, 0, 'TATA') in res
assert (6, 1, 'TATG') in res
print('✓ Problem 15 passed!')


## 5. Summary & Problem Classification Table

| Problem | Bioinformatics Domain | Core Python Concepts | Algorithmic Complexity |
|---|---|---|---|
| **1. Protein Charge** | Proteomics / Isoelectric focus | Set membership, conditionals | $O(N)$ linear scan |
| **2. Enzyme Classifier** | Sequence Annotation | Regular expressions (`re.search`) | Pattern DFA search |
| **3. Sliding Window GC** | Epigenomics / Isochores | Strided loops, slicing | $O(N)$ sliding window |
| **4. K-mers across Range** | Motif Discovery | Dict accumulation, nested loops | $O(K \times N)$ |
| **5. Non-Overlapping Motif**| Restriction Mapping | `str.find()`, pointer jumping | $O(N)$ search |
| **6. Ts/Tv Ratio** | Evolutionary Genetics | `zip()`, set classification | $O(N)$ parallel scan |
| **7. Restriction Palindromes**| Molecular Cloning | Reverse complement, string slicing| $O(L \times N)$ |
| **8. Frameshift Detection** | Mutation Diagnostics | Index divergence, triplet modulo | $O(N)$ |
| **9. PFM & Consensus** | Transcriptional Regulation | 2D matrices, column maximization | $O(M \times L)$ |
| **10. 6-Frame ORF Finder** | Genome Annotation | Reverse complement, nested search | $O(6 \times N)$ |
| **11. Hydropathy Profile** | Membrane Biophysics | Sliding window, float scale lookup | $O(N)$ |
| **12. Triplet Repeats** | Clinical Genetics / Huntington | String matching, tandem counter | $O(N)$ |
| **13. RSCU Codon Bias** | Molecular Evolution | Counter, synonymous grouping | $O(N)$ |
| **14. Distance Matrix** | Phylogenetics | Nested loops, `zip` distance | $O(M^2 \times L)$ |
| **15. Mismatch Motif Search**| Transcription Factor Binding | Fuzzy sliding window, Hamming | $O(L \times N)$ |


## 6. Further Reading & References

1. **Kyte, J., & Doolittle, R. F. (1982)**. A simple method for displaying the hydropathic character of a protein. *Journal of Molecular Biology*, 157(1), 105-132.
2. **Sharp, P. M., & Li, W. H. (1987)**. The codon Adaptation Index—a measure of directional synonymous codon usage bias. *Nucleic Acids Research*, 15(3), 1281-1295.
3. **PROSITE Database**: https://prosite.expasy.org/
4. **Rosalind Bioinformatics Stronghold**: https://rosalind.info/
